In [ ]:
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv() 

True

In [30]:
prompt1 = PromptTemplate(
    template="Generate a short notes from following text. \n{text}",
    input_variables=['text']
)

prompt2 = PromptTemplate(
    template="Generate a quiz from following notes. \n{text}",
    input_variables=['text']
)

prompt3 = PromptTemplate(
    template="Merge the the following notes and quiz into one single document. Answer in plain text. Do not use markdown. \nnotes - > {notes} \n 'quiz -> {quiz}",
    input_variables={'notes', 'quiz'}
)

In [31]:
model1 = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite"
)

model2 = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

In [ ]:
output_parser = StrOutputParser() 

In [33]:
parallel_chain = RunnableParallel(
   { 
       'notes': prompt1 | model1 | output_parser,
        'quiz': prompt2 | model2 | output_parser
    }
)

merge_chain = prompt3 | model1 | output_parser

final_chain = parallel_chain | merge_chain

In [ ]:
text = """Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function (called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified for the decision function. Common kernels are provided, but it is also possible to specify custom kernels.

The disadvantages of support vector machines include:

If the number of features is much greater than the number of samples, avoid over-fitting in choosing Kernel functions and regularization term is crucial.

SVMs do not directly provide probability estimates, these are calculated using an expensive five-fold cross-validation (see Scores and probabilities, below).

The support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input. However, to use an SVM to make predictions for sparse data, it must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64.

""" 

In [ ]:
final_chain.invoke({'text': text}) 

'Notes: Support Vector Machines (SVMs)\n\nOverview\nPurpose: Supervised learning for classification, regression, and outlier detection.\n\nAdvantages\nHigh Dimensionality: Effective even when dimensions exceed the number of samples.\nMemory Efficiency: Uses only a subset of training points ("support vectors") for the decision function.\nVersatility: Supports various built-in and custom Kernel functions.\n\nDisadvantages\nOverfitting Risk: Requires careful selection of Kernels and regularization when features significantly outnumber samples.\nProbability Estimates: Does not provide direct probability estimates; requires expensive five-fold cross-validation.\n\nImplementation Details (scikit-learn)\nInput Types: Supports both dense (numpy.ndarray) and sparse (scipy.sparse) vectors.\nConstraint: Models must be fit on the same data type (sparse vs. dense) used for prediction.\nPerformance Tip: Use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64 for op

In [ ]:
final_chain.get_graph().print_ascii() 

                    +---------------------------+                      
                    | Parallel<notes,quiz>Input |                      
                    +---------------------------+                      
                       ***                   ***                       
                   ****                         ****                   
                 **                                 **                 
    +----------------+                          +----------------+     
    | PromptTemplate |                          | PromptTemplate |     
    +----------------+                          +----------------+     
             *                                           *             
             *                                           *             
             *                                           *             
+------------------------+                  +------------------------+ 
| ChatGoogleGenerativeAI |                  | ChatGoogleGenerati